In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import col, count, when, isnan, row_number, concat, lit, lpad

catalog_name = 'automobilerepair'

In [0]:
df = spark.read.table("automobilerepair.bronze.stg_order")

In [0]:
display(df.limit(5))

In [0]:
row_count = df.count()
row_count

In [0]:
# DUPLICATE ANALYSIS
print("\nDUPLICATE ANALYSIS")
duplicate_order_ids = df.groupBy("order_id").count().filter(col("count") > 1)
print(f"Duplicate order_ids: {duplicate_order_ids.count()}")
df = df.dropDuplicates(["order_id"])
row_count = df.count()
print(f"Number of rows after removing duplicates: {row_count}")

In [0]:
print("NULL VALUE ANALYSIS")
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df.columns
])
print("Null counts by column:")
display(null_counts)

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("technician_name").orderBy(f.col("technician_id").desc())
df = df.withColumn(
    "technician_id",
    f.first("technician_id", ignorenulls=True).over(window_spec)
)

null_count_technician_id = df.filter(col("technician_id").isNull()).count()
print("Technician_id null count")
null_count_technician_id


df = df.withColumn(
    "vehicle_model",
    f.when(col("vehicle_model").isNull(), "not mentioned").otherwise(col("vehicle_model"))
)


In [0]:
df = df.withColumn(
    "work_started_flag",
    when(col("actual_work_start_datetime").isNotNull(), 1).otherwise(0)
).withColumn(
    "work_completed_flag",
    when(col("actual_completion_datetime").isNotNull(), 1).otherwise(0)
).withColumn(
    "delivered_flag",
    when(col("actual_delivery_datetime").isNotNull(), 1).otherwise(0)
)

In [0]:
df = df.withColumnRenamed("_modified", "modified")

In [0]:
df_customer = df.select(
    "customer_name",
    "customer_phone", 
    "modified",
    "inserted_at",
    "updated_at"
).dropDuplicates()

window_spec = Window.orderBy("customer_name", "customer_phone")
df_customer = df_customer.withColumn(
    "customer_id",
    concat(
        lit("CUST"),
        lpad(row_number().over(window_spec), 7, "0")
    )
)
display(df_customer)

In [0]:
df_technician = df.select(
    "technician_id",
    "technician_name",
    "modified",
    "inserted_at",
    "updated_at"
).dropDuplicates()

display(df_technician)

In [0]:
df = df.join(
    df_customer.select("customer_name", "customer_phone", "customer_id"),
    on=["customer_name", "customer_phone"],
    how="left"
).drop("customer_name", "customer_phone", "technician_name")

display(df)

In [0]:
df_vehicle = df.select(
    "vehicle_no",
    "vehicle_make",
    "vehicle_model",
    "modified",
    "inserted_at",
    "updated_at"
).dropDuplicates()

df_vehicle = df_vehicle.withColumnRenamed("vehicle_no", "vehicle_id")
display(df_vehicle)

df = df.withColumnRenamed("vehicle_no", "vehicle_id") \
       .drop("vehicle_make", "vehicle_model")

In [0]:
df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.silver.slv_order")
df_customer.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.silver.slv_customer")
df_technician.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.silver.slv_technician")
df_vehicle.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.silver.slv_vehicle")